# 상품 분류 파인튜닝 — 최종 점검

튜닝은 끝났다. 이 노트북은 **결론 재확인 + 남은 두 가지 점검**만 한다 — 기존 세션에
안 이어도 된다(전부 디스크에 저장된 산출물만 읽는다). `../CLASSIFIER_FINETUNE_BRIEF.md`,
`aihub_pipeline_server.ipynb`(전체 파이프라인 기록)와 같이 참고.

In [ ]:
# --- 환경 설정 ---
import json
import subprocess
import sys
import threading
import time
from pathlib import Path


def run(script, *args, heartbeat_every=30):
    """스크립트를 실시간 스트리밍으로 실행한다. GPU 작업이 죽었는지 살았는지
    구분하려고 heartbeat를 같이 찍는다."""
    cmd = [sys.executable, str(HERE / script), *[str(a) for a in args]]
    print("$ " + " ".join(cmd) + "\n")
    start = time.time()
    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", bufsize=1,
    )
    stop_heartbeat = threading.Event()

    def heartbeat():
        while not stop_heartbeat.wait(heartbeat_every):
            print(f"  ... 아직 실행 중 (경과 {int(time.time() - start)}초)")

    thread = threading.Thread(target=heartbeat, daemon=True)
    thread.start()
    for line in process.stdout:
        print(f"[{time.time() - start:7.1f}s] {line}", end="")
    process.wait()
    stop_heartbeat.set()
    elapsed = time.time() - start
    status = "성공" if process.returncode == 0 else f"⚠ 실패(returncode={process.returncode})"
    print(f"\n{status} — 총 {elapsed:.1f}초")
    return process


HERE = Path("/home/j-i15a108/work/S15P11A108/AI/validation/CLASS/product-info-finetune")
assert (HERE / "aihub_dataset.py").exists(), f"HERE가 틀렸다: {HERE}"
ARTIFACTS = HERE / "artifacts"
VARIANT = "masked"
print("good")

## 1. 지금까지 결론

- **카테고리**: last-N=4(비전 인코더 마지막 4블록 언프리즈) — test **0.8424**
- **색상**: LoRA r=16·alpha=32, 카테고리 체크포인트에서 이어받은 순차학습 — test **0.7770**
- 백본 하나 공유(순차학습된 backbone으로 양쪽 다 서빙)는 폐기 — sklearn 프로브로 확인한
  결과 색상-적응 backbone은 카테고리에서 zero-shot(0.6763)보다도 낮았다
  (last-N 0.6495, LoRA 0.6217). **두 모델을 따로 배포한다.**

In [ ]:
run(
    "aihub_finetune_summary.py",
    "--artifacts-finetune-dir", HERE / "artifacts_finetune",
    "--best-per-group", "method,task",
    "--sort-by", "test_accuracy",
    "--out", HERE / "artifacts_finetune" / "leaderboard_final.md",
)

## 2. 남은 점검 ① — 미커버 24종(신발·가방 등) zero-shot이 last-N=4 백본에서도 유지되는가

우리 정답셋(`product-info-check/inputs/` + `data/ground_truth.json`)에서 last-N=4 헤드가
모르는 클래스(=미커버 클래스)만 골라, 원본(동결) backbone과 last-N=4 backbone 각각으로
zero-shot 정확도를 재서 비교한다. 색상 순차학습이 카테고리를 zero-shot 이하로 깎아먹는 걸
이미 확인했으니(`aihub_finetune_probe.py`), 카테고리 학습도 미커버 클래스를 깎아먹었을
가능성을 같은 방식으로 확인하는 것이다.

In [ ]:
run(
    "aihub_ground_truth_zero_shot_check.py",
    "--ground-truth-dir", HERE.parent / "product-info-check",
    "--checkpoint", HERE / "artifacts_finetune" / "vit_last4_category_masked_lr1e-05" / "checkpoint.pt",
    "--method", "last-n",
    "--unfreeze-layers", 4,
)

## 3. 남은 점검 ② — 지연시간 실측

last-N=4·LoRA-웜 각각 forward 시간, 두 모델 순차 실행 시 합산 시간을 잰다.
`product-info-check/latency.py` 하네스 재사용 예정 — 스크립트 준비되면 이 셀에 채운다.

In [ ]:
# TODO: 지연시간 실측 — 스크립트 준비되면 채움

## 4. 산출물 확인

In [ ]:
for path in sorted(ARTIFACTS.glob("*")):
    size = path.stat().st_size if path.is_file() else 0
    print(f"{size / 1e6:10.2f} MB  {path.name}")

for meta_file in sorted((HERE / "artifacts_finetune").glob("*/meta.json")):
    print("\n--- " + str(meta_file.relative_to(HERE)) + " ---")
    print(json.dumps(json.loads(meta_file.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))